# CAPTCHA Farm Attack Simulation

This notebook simulates a CAPTCHA farm attack scenario to test the traceable token mechanism:

**Attack Scenario:**
1. **Step 1**: Simulate N legitimate challenges solved by a "worker" to obtain traceable tokens T_W
2. **Step 2**: For each token, have an "attacker" script try to reuse it from a different session + device fingerprint:
   - Different IP (use override in payload)
   - Different user agent / fingerprint
3. **Step 3**: Record whether the server accepts or rejects each reuse attempt

**Metrics:**
- For varying N (e.g., N = 100, 500, 1000):
  - N Tokens | Reuse Attempts | # Accepted | Success Rate (%)


## Setup and Imports


In [1]:
import hashlib
import time
import os
import json
import sqlite3
import base64
from PIL import Image
import io
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric.utils import Prehashed
from cryptography.hazmat.primitives import serialization
from cryptography.exceptions import InvalidSignature
import platform
import uuid
import datetime
import jwt
import hmac
from cryptography.fernet import Fernet
import calendar
import random

# Import thermal image checker
try:
    from thermalImageChecker import is_thermal_image
except ImportError:
    print("Warning: thermalImageChecker not found. Using mock function.")
    def is_thermal_image(image):
        return "Thermal Image"  # Mock function for testing

print("Imports completed successfully.")


Imports completed successfully.


## Configuration


In [2]:
# Configuration
DB_PATH = 'thermal_captcha.db'
TEST_IMAGE_PATH = './25.bmp'  # Adjust path as needed
CHALLENGE_TTL_SECONDS = 120  # seconds

# Test parameters
SITE_KEY = "28024d2408754883a7eff1db9cea2b00"
WEBSITE_NAME = "microsoft.com"
WORKER_IP = "192.168.1.100"  # Worker's IP address
ATTACKER_IP_BASE = "10.0.0."  # Attacker's IP base (will vary)

# Server secret key
SERVER_SECRET_KEY = '9a61f13c92a04010b0f87a15734673c8'.encode()

# Test values for N
TEST_VALUES = [100, 500, 1000]

print(f"Configuration loaded. Testing with N = {TEST_VALUES}")


Configuration loaded. Testing with N = [100, 500, 1000]


## Helper Functions


In [3]:
def generate_device_fingerprint(user_ip, user_agent=None, device_public_key_pem=None):
    """Generate a public-key-bound device fingerprint."""
    fingerprint_data = {
        'ip': user_ip,
        'user_agent': user_agent or 'Unknown',
        'device_public_key_sha256': hashlib.sha256((device_public_key_pem or '').encode()).hexdigest()
    }
    fingerprint_string = json.dumps(fingerprint_data, sort_keys=True)
    device_fingerprint = hashlib.sha256(fingerprint_string.encode()).hexdigest()
    return device_fingerprint

def canonicalize_challenge(challenge):
    return json.dumps(challenge, sort_keys=True, separators=(",", ":")).encode()


def build_signed_blob(challenge, binary_data, device_fingerprint):
    return canonicalize_challenge(challenge) + binary_data + device_fingerprint.encode()


def create_valid_payload(image_path, issued_at=None, challenge_nonce=None, device_fingerprint=None, user_ip=None, user_agent=None):
    """Create a valid challenge-bound payload with optional fields."""
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()

    try:
        with open(image_path, "rb") as image_file:
            image_binary = image_file.read()
    except FileNotFoundError:
        print(f"Warning: {image_path} not found. Using mock image data.")
        image_binary = b"mock_thermal_image_data" * 100

    if issued_at is None:
        issued_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    if challenge_nonce is None:
        challenge_nonce = os.urandom(16).hex()

    if device_fingerprint is None:
        if user_ip is None:
            user_ip = WORKER_IP

    public_key_base64 = public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    ).decode()

    if device_fingerprint is None:
        device_fingerprint = generate_device_fingerprint(
            user_ip,
            user_agent,
            device_public_key_pem=public_key_base64
        )

    challenge = {
        "challenge_id": str(uuid.uuid4()),
        "nonce": challenge_nonce,
        "issued_at": issued_at,
        "expires_in_seconds": CHALLENGE_TTL_SECONDS
    }

    signed_blob = build_signed_blob(challenge, image_binary, device_fingerprint)
    binary_hash = hashlib.sha256(signed_blob).digest()
    signature = private_key.sign(
        binary_hash,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        Prehashed(hashes.SHA256())
    )

    thermal_image_base64 = base64.b64encode(image_binary).decode()
    signature_base64 = base64.b64encode(signature).decode()
    payload = {
        "user_IP": user_ip if user_ip else WORKER_IP,
        "SiteKey": SITE_KEY,
        "WebsiteName": WEBSITE_NAME,
        "challenge": challenge,
        "thermal_image": thermal_image_base64,
        "signature": signature_base64,
        "public_key": public_key_base64,
        "device_fingerprint": device_fingerprint
    }

    return json.dumps(payload), device_fingerprint

print("Helper functions loaded.")


Helper functions loaded.


## Server-Side Functions


In [4]:
def is_user_nonce_present(nonce, cursor):
    """Check if nonce has been used before"""
    cursor.execute('SELECT * FROM UserNonce WHERE Nonce = ?', (nonce,))
    return cursor.fetchone() is not None

def add_user_nonce(user_id, nonce, cursor, conn):
    """Add nonce to database"""
    cursor.execute('INSERT INTO UserNonce (UserID, Nonce) VALUES (?, ?)', (user_id, nonce))
    conn.commit()

def add_user(site_key, website_name, user_ip, cursor, conn):
    """Add user to database"""
    cursor.execute('SELECT * FROM ServerVSWebsite WHERE SiteKey = ?', (site_key,))
    if cursor.fetchone() is None:
        return None
    
    cursor.execute('''
        INSERT INTO Users (SiteKey, WebsiteName, UserIPAddress)
        VALUES (?, ?, ?)
    ''', (site_key, website_name, user_ip))
    conn.commit()
    return cursor.lastrowid

def add_detection(user_id, result, cursor, conn):
    """Add detection result to database"""
    if user_id is None:
        return
    cursor.execute('INSERT INTO Detections (UserID, Result) VALUES (?, ?)', (user_id, result))
    conn.commit()

def add_session_tracking(user_id, session_id, device_fingerprint, cursor, conn):
    """Store sessionID and device fingerprint for traceable token verification"""
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS SessionTracking (
            UniqueID INTEGER PRIMARY KEY AUTOINCREMENT,
            UserID INTEGER NOT NULL,
            SessionID TEXT NOT NULL UNIQUE,
            DeviceFingerprint TEXT NOT NULL,
            Timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (UserID) REFERENCES Users(UniqueID)
        )
    ''')
    
    cursor.execute('''
        INSERT INTO SessionTracking (UserID, SessionID, DeviceFingerprint)
        VALUES (?, ?, ?)
    ''', (user_id, session_id, device_fingerprint))
    
    conn.commit()

def get_session_tracking(user_id, session_id, cursor):
    """Retrieve stored sessionID and device fingerprint for verification"""
    cursor.execute('''
        SELECT SessionID, DeviceFingerprint FROM SessionTracking 
        WHERE UserID = ? AND SessionID = ?
    ''', (user_id, session_id))
    row = cursor.fetchone()
    if row:
        return {'sessionID': row[0], 'device_fingerprint': row[1]}
    return None

def get_shared_key_by_sitekey(site_key, cursor):
    """Get shared key for a site key"""
    cursor.execute('SELECT SharedKey FROM ServerVSWebsite WHERE SiteKey = ?', (site_key,))
    row = cursor.fetchone()
    if row:
        return row[0]
    return None

def derive_public_key_bound_fingerprint(user_ip, public_key_pem, user_agent=None):
    fingerprint_data = {
        'ip': user_ip,
        'user_agent': user_agent or 'Unknown',
        'device_public_key_sha256': hashlib.sha256((public_key_pem or '').encode()).hexdigest()
    }
    fingerprint_string = json.dumps(fingerprint_data, sort_keys=True)
    return hashlib.sha256(fingerprint_string.encode()).hexdigest()


def validate_challenge_freshness(challenge):
    required_challenge_keys = ["challenge_id", "nonce", "issued_at", "expires_in_seconds"]
    for challenge_key in required_challenge_keys:
        if challenge_key not in challenge:
            raise KeyError(f"Missing required challenge key: {challenge_key}")

    issued_dt_utc = datetime.datetime.strptime(
        challenge["issued_at"], "%Y-%m-%dT%H:%M:%SZ"
    ).replace(tzinfo=datetime.timezone.utc)
    now_dt_utc = datetime.datetime.now(datetime.timezone.utc)
    age_seconds = (now_dt_utc - issued_dt_utc).total_seconds()
    ttl_seconds = int(challenge["expires_in_seconds"])

    if age_seconds < 0:
        raise ValueError(f"Challenge issued in the future (age={age_seconds})")
    if age_seconds > ttl_seconds:
        raise ValueError(f"Challenge expired ({age_seconds} seconds old, ttl={ttl_seconds})")

    return challenge["nonce"], age_seconds


def validate_payload(payload_json, cursor, conn, suppress_output=True):
    """
    Validate a payload and return (accepted, reason, token_info)
    accepted: True if payload is accepted, False otherwise
    reason: String describing why it was accepted/rejected
    token_info: Dictionary with token information if accepted, None otherwise
    """
    try:
        payload = json.loads(payload_json)
        required_keys = ["thermal_image", "challenge", "signature", "public_key", "SiteKey", "WebsiteName", "user_IP", "device_fingerprint"]
        for key in required_keys:
            if key not in payload:
                return False, f"Missing required key: {key}", None

        thermal_image_base64 = payload["thermal_image"]
        challenge = payload["challenge"]
        signature_base64 = payload["signature"]
        public_key_base64 = payload["public_key"]
        site_key = payload["SiteKey"]
        website_name = payload["WebsiteName"]
        user_IP = payload["user_IP"]
        device_fingerprint = payload["device_fingerprint"]

        thermal_image_binary = base64.b64decode(thermal_image_base64)
        received_signature = base64.b64decode(signature_base64)
        public_key = serialization.load_pem_public_key(public_key_base64.encode())

        expected_device_fingerprint = derive_public_key_bound_fingerprint(
            user_IP,
            public_key_base64
        )
        if device_fingerprint != expected_device_fingerprint:
            return False, "Device fingerprint does not match public-key-bound fingerprint", None

        # Freshness + replay checks run before model inference
        try:
            challenge_nonce, _ = validate_challenge_freshness(challenge)
        except (KeyError, ValueError) as e:
            return False, str(e), None

        if is_user_nonce_present(challenge_nonce, cursor):
            return False, "Nonce already used", None

        # Verify signature over challenge || image || fingerprint
        signed_blob = build_signed_blob(challenge, thermal_image_binary, device_fingerprint)
        signed_blob_hash = hashlib.sha256(signed_blob).digest()
        try:
            public_key.verify(
                received_signature,
                signed_blob_hash,
                padding.PSS(
                    mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH
                ),
                Prehashed(hashes.SHA256())
            )
        except InvalidSignature:
            return False, "Invalid signature", None
        except Exception as e:
            return False, f"Signature verification error: {str(e)}", None

        # Thermal check only after cryptographic checks pass
        try:
            image = Image.open(io.BytesIO(thermal_image_binary))
            result = is_thermal_image(image)
            if result != "Thermal Image":
                return False, f"Not a thermal image: {result}", None
        except Exception as e:
            return False, f"Image processing error: {str(e)}", None

        user_id = add_user(site_key, website_name, user_IP, cursor, conn)
        if user_id is not None:
            add_detection(user_id, 85, cursor, conn)
            add_user_nonce(user_id, challenge_nonce, cursor, conn)

            session_id = str(uuid.uuid4())
            add_session_tracking(user_id, session_id, device_fingerprint, cursor, conn)

            token_info = {
                'user_id': user_id,
                'session_id': session_id,
                'device_fingerprint': device_fingerprint
            }

            return True, "Accepted", token_info

        return False, "Failed to add user", None

    except Exception as e:
        return False, f"Processing error: {str(e)}", None

def derive_combined_key(server_key, shared_key):
    """Derive combined key for dual-layered encryption"""
    return base64.urlsafe_b64encode(hmac.new(server_key, shared_key.encode(), digestmod='sha256').digest())

def generate_token(user_id, site_key, session_id, device_fingerprint):
    """Generate traceable JWT token with sessionID and device fingerprint"""
    expiration_time = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=2)
    nonce = os.urandom(16).hex()
    token_payload = {
        'exp': expiration_time,
        'result_row_number': user_id,
        'SiteKey': site_key,
        'nonce': nonce,
        'sessionID': session_id,
        'device_fingerprint': device_fingerprint
    }
    token = jwt.encode(token_payload, SERVER_SECRET_KEY, algorithm='HS256')
    return token

def encrypt_token(token, shared_key):
    """Encrypt token using Fernet with combined key"""
    combined_key = derive_combined_key(SERVER_SECRET_KEY, shared_key)
    fernet_combined = Fernet(combined_key)
    if isinstance(token, str):
        token = token.encode()
    encrypted_token = fernet_combined.encrypt(token)
    return encrypted_token

def decrypt_token(encrypted_token, shared_key):
    """Decrypt token using Fernet with combined key"""
    combined_key = derive_combined_key(SERVER_SECRET_KEY, shared_key)
    fernet_combined = Fernet(combined_key)
    decrypted_token = fernet_combined.decrypt(encrypted_token).decode()
    return decrypted_token

def is_token_nonce_present(nonce, cursor):
    """Check if token nonce has been used"""
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS TokenNonce (
            UniqueID INTEGER PRIMARY KEY AUTOINCREMENT,
            Token INTEGER,
            Nonce TEXT NOT NULL UNIQUE
        )
    ''')
    cursor.execute('SELECT * FROM TokenNonce WHERE Nonce = ?', (nonce,))
    return cursor.fetchone() is not None

def add_token_nonce(user_id, nonce, cursor, conn):
    """Add token nonce to database"""
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS TokenNonce (
            UniqueID INTEGER PRIMARY KEY AUTOINCREMENT,
            Token INTEGER,
            Nonce TEXT NOT NULL UNIQUE
        )
    ''')
    cursor.execute('INSERT INTO TokenNonce (Token, Nonce) VALUES (?, ?)', (user_id, nonce))
    conn.commit()

def validate_traceable_token(encrypted_token, shared_key, cursor, conn, current_device_fingerprint=None):
    """
    Validate traceable token by checking sessionID and device fingerprint integrity.
    Also checks current request's device fingerprint against token's device fingerprint.
    Returns (is_valid, reason, user_id)
    """
    try:
        token = decrypt_token(encrypted_token, shared_key)
        decoded = jwt.decode(token, SERVER_SECRET_KEY, algorithms=['HS256'])
        
        # Check if token nonce has been used
        received_token_nonce = decoded.get('nonce')
        if is_token_nonce_present(received_token_nonce, cursor):
            return False, "Replay attack detected: nonce already used", None
        
        user_id = decoded.get('result_row_number')
        received_session_id = decoded.get('sessionID')
        received_device_fingerprint = decoded.get('device_fingerprint')
        
        if not received_session_id or not received_device_fingerprint:
            return False, "Missing sessionID or device fingerprint", None
        
        # Retrieve stored session tracking data
        stored_session = get_session_tracking(user_id, received_session_id, cursor)
        
        if not stored_session:
            return False, "SessionID not found or token tampered with", None
        
        # Verify device fingerprint in token matches stored one
        if stored_session['device_fingerprint'] != received_device_fingerprint:
            return False, "Device fingerprint mismatch - token may be relayed or tampered with", None
        
        # Verify sessionID matches
        if stored_session['sessionID'] != received_session_id:
            return False, "SessionID mismatch - token may be tampered with", None
        
        # CRITICAL: Check current request's device fingerprint against token's device fingerprint
        # This prevents token reuse from different devices (CAPTCHA farm attack)
        if current_device_fingerprint is not None:
            if current_device_fingerprint != received_device_fingerprint:
                return False, "Device fingerprint mismatch - current request device does not match token device (CAPTCHA farm attack detected)", None
        
        # Mark token nonce as used
        add_token_nonce(user_id, received_token_nonce, cursor, conn)
        
        return True, "Token validated successfully", user_id
        
    except jwt.ExpiredSignatureError:
        return False, "Token has expired", None
    except jwt.InvalidTokenError:
        return False, "Invalid token", None
    except Exception as e:
        return False, f"Token validation error: {str(e)}", None

print("Server-side functions loaded.")


Server-side functions loaded.


In [5]:
def worker_solve_challenge(image_path, worker_ip, worker_user_agent, cursor, conn):
    """
    Simulate a worker solving a legitimate CAPTCHA challenge.
    Returns: (success, token_info, encrypted_token) or (False, None, None)
    """
    # Create valid payload with worker's device fingerprint
    payload_json, device_fingerprint = create_valid_payload(
        image_path, 
        user_ip=worker_ip,
        user_agent=worker_user_agent
    )
    
    # Validate payload (server-side)
    accepted, reason, token_info = validate_payload(payload_json, cursor, conn, suppress_output=True)
    
    if accepted and token_info:
        # Generate token
        shared_key = get_shared_key_by_sitekey(SITE_KEY, cursor)
        if shared_key:
            token = generate_token(
                token_info['user_id'],
                SITE_KEY,
                token_info['session_id'],
                token_info['device_fingerprint']
            )
            encrypted_token = encrypt_token(token, shared_key)
            encrypted_token_base64 = base64.b64encode(encrypted_token).decode()
            
            return True, token_info, encrypted_token_base64
    
    return False, None, None

print("Worker function defined.")


Worker function defined.


## Step 2: Attacker Function - Try to Reuse Tokens


In [6]:
def attacker_reuse_token(encrypted_token_base64, attacker_ip, attacker_user_agent, cursor, conn):
    """
    Simulate an attacker trying to reuse a token from a different session/device.
    The attacker uses a different IP and user agent, which results in a different device fingerprint.
    Returns: (accepted, reason)
    """
    try:
        # Get shared key
        shared_key = get_shared_key_by_sitekey(SITE_KEY, cursor)
        if not shared_key:
            return False, "Shared key not found"
        
        # Generate attacker's device fingerprint (different from worker's)
        attacker_device_fingerprint = generate_device_fingerprint(attacker_ip, attacker_user_agent)
        
        # Try to validate the token (this will check sessionID, device fingerprint, and current device fingerprint)
        is_valid, reason, user_id = validate_traceable_token(
            base64.b64decode(encrypted_token_base64),
            shared_key,
            cursor,
            conn,
            current_device_fingerprint=attacker_device_fingerprint  # Pass attacker's device fingerprint
        )
        
        return is_valid, reason
        
    except Exception as e:
        return False, f"Error: {str(e)}"

print("Attacker function defined.")


Attacker function defined.


## Main Simulation Function


In [7]:
def simulate_captcha_farm_attack(n_tokens, image_path, db_path):
    """
    Simulate CAPTCHA farm attack scenario:
    1. Worker solves N legitimate challenges to obtain tokens
    2. Attacker tries to reuse each token with different device fingerprint
    3. Record acceptance/rejection rates
    
    Returns: (num_tokens, reuse_attempts, num_accepted, success_rate)
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Worker configuration
    worker_ip = WORKER_IP
    worker_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    
    # Attacker configuration (different from worker)
    attacker_user_agents = [
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
        "Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X) AppleWebKit/605.1.15",
    ]
    
    tokens_collected = []
    accepted_count = 0
    rejected_count = 0
    
    print(f"\n=== Simulating CAPTCHA Farm Attack with N = {n_tokens} ===")
    print(f"Step 1: Worker solving {n_tokens} legitimate challenges...")
    
    # Step 1: Worker solves N challenges
    for i in range(n_tokens):
        # Generate different attacker IP for each attempt
        attacker_ip = f"{ATTACKER_IP_BASE}{random.randint(1, 254)}"
        attacker_user_agent = random.choice(attacker_user_agents)
        
        success, token_info, encrypted_token = worker_solve_challenge(
            image_path, worker_ip, worker_user_agent, cursor, conn
        )
        
        if success and encrypted_token:
            tokens_collected.append({
                'token': encrypted_token,
                'token_info': token_info,
                'attacker_ip': attacker_ip,
                'attacker_user_agent': attacker_user_agent
            })
        
        # Progress update
        if (i + 1) % max(1, n_tokens // 10) == 0:
            print(f"  Worker progress: {i + 1}/{n_tokens} challenges solved")
    
    print(f"Step 1 complete: Collected {len(tokens_collected)} tokens")
    print(f"Step 2: Attacker attempting to reuse {len(tokens_collected)} tokens...")
    
    # Step 2: Attacker tries to reuse each token
    for i, token_data in enumerate(tokens_collected):
        accepted, reason = attacker_reuse_token(
            token_data['token'],
            token_data['attacker_ip'],
            token_data['attacker_user_agent'],
            cursor,
            conn
        )
        
        if accepted:
            accepted_count += 1
        else:
            rejected_count += 1
        
        # Progress update
        if (i + 1) % max(1, len(tokens_collected) // 10) == 0:
            print(f"  Attacker progress: {i + 1}/{len(tokens_collected)} reuse attempts (Accepted: {accepted_count}, Rejected: {rejected_count})")
    
    conn.close()
    
    # Calculate success rate
    reuse_attempts = len(tokens_collected)
    success_rate = (accepted_count / reuse_attempts * 100) if reuse_attempts > 0 else 0.0
    
    return n_tokens, reuse_attempts, accepted_count, success_rate

print("Simulation function defined.")


Simulation function defined.


In [8]:
# Run simulation for each N value
results = []

for n in TEST_VALUES:
    print(f"\n{'='*70}")
    print(f"Running simulation for N = {n}")
    print(f"{'='*70}")
    
    num_tokens, reuse_attempts, num_accepted, success_rate = simulate_captcha_farm_attack(
        n, TEST_IMAGE_PATH, DB_PATH
    )
    
    results.append({
        'N': num_tokens,
        'Reuse Attempts': reuse_attempts,
        'Accepted': num_accepted,
        'Success Rate (%)': success_rate
    })
    
    print(f"\nResults for N = {n}:")
    print(f"  Tokens Collected: {num_tokens}")
    print(f"  Reuse Attempts: {reuse_attempts}")
    print(f"  Accepted: {num_accepted}")
    print(f"  Success Rate: {success_rate:.1f}%")

print(f"\n{'='*70}")
print("All simulations completed!")
print(f"{'='*70}")



Running simulation for N = 100

=== Simulating CAPTCHA Farm Attack with N = 100 ===
Step 1: Worker solving 100 legitimate challenges...
  Worker progress: 10/100 challenges solved
  Worker progress: 20/100 challenges solved
  Worker progress: 30/100 challenges solved
  Worker progress: 40/100 challenges solved
  Worker progress: 50/100 challenges solved
  Worker progress: 60/100 challenges solved
  Worker progress: 70/100 challenges solved
  Worker progress: 80/100 challenges solved
  Worker progress: 90/100 challenges solved
  Worker progress: 100/100 challenges solved
Step 1 complete: Collected 100 tokens
Step 2: Attacker attempting to reuse 100 tokens...
  Attacker progress: 10/100 reuse attempts (Accepted: 0, Rejected: 10)
  Attacker progress: 20/100 reuse attempts (Accepted: 0, Rejected: 20)
  Attacker progress: 30/100 reuse attempts (Accepted: 0, Rejected: 30)
  Attacker progress: 40/100 reuse attempts (Accepted: 0, Rejected: 40)
  Attacker progress: 50/100 reuse attempts (Accep

## Results Summary Table


In [9]:
# Print results in the requested format
print("\n" + "="*70)
print("CAPTCHA Farm Attack Simulation Results")
print("="*70)
print("\nN Tokens | Reuse Attempts | # Accepted | Success Rate (%)")
print("-" * 70)

for result in results:
    n = result['N']
    reuse = result['Reuse Attempts']
    accepted = result['Accepted']
    rate = result['Success Rate (%)']
    print(f"{n:>8} | {reuse:>14} | {accepted:>10} | {rate:>15.1f}")

print("-" * 70)

# Summary
total_attempts = sum(r['Reuse Attempts'] for r in results)
total_accepted = sum(r['Accepted'] for r in results)
overall_rate = (total_accepted / total_attempts * 100) if total_attempts > 0 else 0.0

print(f"\nTotal Reuse Attempts: {total_attempts}")
print(f"Total Accepted: {total_accepted}")
print(f"Overall Success Rate: {overall_rate:.2f}%")

if total_accepted == 0:
    print("\n✓ All token reuse attempts were successfully rejected!")
    print("✓ Traceable token mechanism is working correctly.")
    print("✓ CAPTCHA farm attack prevented!")
else:
    print(f"\n⚠ WARNING: {total_accepted} token reuse attempts were accepted!")
    print("⚠ Review traceable token mechanism.")

print("="*70)



CAPTCHA Farm Attack Simulation Results

N Tokens | Reuse Attempts | # Accepted | Success Rate (%)
----------------------------------------------------------------------
     100 |            100 |          0 |             0.0
     500 |            500 |          0 |             0.0
    1000 |           1000 |          0 |             0.0
----------------------------------------------------------------------

Total Reuse Attempts: 1600
Total Accepted: 0
Overall Success Rate: 0.00%

✓ All token reuse attempts were successfully rejected!
✓ Traceable token mechanism is working correctly.
✓ CAPTCHA farm attack prevented!


## Detailed Results (Optional DataFrame)


In [10]:
try:
    import pandas as pd
    
    df_results = pd.DataFrame(results)
    print("\nDetailed Results DataFrame:")
    print(df_results.to_string(index=False))
    
except ImportError:
    print("Pandas not available. Skipping DataFrame display.")
    print("\nResults summary:")
    for result in results:
        print(f"  N={result['N']}: {result['Reuse Attempts']} attempts, {result['Accepted']} accepted, {result['Success Rate (%)']:.1f}% success rate")



Detailed Results DataFrame:
   N  Reuse Attempts  Accepted  Success Rate (%)
 100             100         0               0.0
 500             500         0               0.0
1000            1000         0               0.0
